##### Copyright 2026 Google LLC.

In [2]:
#@title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Use Gemini thinking

> **Note:** This notebook uses the Interactions API, the latest way to interact with Gemini models. Looking for the `generateContent` version? Check the [archive branch](https://github.com/google-gemini/cookbook/blob/archive/generate-content-api/quickstarts/Get_started_thinking.ipynb).

<a target="_blank" href="https://colab.research.google.com/github/google-gemini/cookbook/blob/main/quickstarts/Get_started_thinking.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" height=30/></a>

All Gemini models are trained to do a [thinking process](https://ai.google.dev/gemini-api/docs/thinking-mode) (or reasoning) before getting to a final answer. As a result, those models usually get better results on harder tasks that require multiple processing steps: complex math, coding, reasoning over multi-step instructions and multimodal understanding.

While thinking is always on, you can configure the amount of thinking the model does by using **thinking levels** (`minimal`, `low`, `medium`, `high`). This lets you balance between response speed/cost and reasoning depth depending on your use case.

## Understanding thinking models

Thinking models are optimized for complex tasks that need multiple rounds of strategizing and iteratively solving.

You can control the thinking effort using **thinking levels**:

| Thinking Level | Use Case |
|---|---|
| **High** (default) | Complex reasoning, math, coding, multi-step problems |
| **Medium** | Good balance between speed and reasoning |
| **Low** | Faster responses with some reasoning |
| **Minimal** | Fastest responses, minimal reasoning (roughly equivalent to "off") |

You can set the thinking level using `types.ThinkingConfig(thinking_level=types.ThinkingLevel.HIGH)`.

## Setup

This section install the SDK, set it up using your [API key](../quickstarts/Authentication.ipynb), imports the relevant libs, downloads the sample videos and upload them to Gemini.

Just collapse (click on the little arrow on the left of the title) and run this section if you want to jump straight to the examples (just don't forget to run it otherwise nothing will work).

### Install SDK

The **[Google Gen AI SDK](https://ai.google.dev/gemini-api/docs/sdks)** provides programmatic access to Gemini models using both the [Google AI for Developers](https://ai.google.dev/gemini-api/docs) and [Vertex AI](https://cloud.google.com/vertex-ai/generative-ai/docs/overview) APIs. With a few exceptions, code that runs on one platform will run on both. This means that you can prototype an application using the Developer API and then migrate the application to Vertex AI without rewriting your code.

More details about this new SDK on the [documentation](https://ai.google.dev/gemini-api/docs/sdks) or in the [Getting started ![image](https://storage.googleapis.com/generativeai-downloads/images/colab_icon16.png)](../quickstarts/Get_started.ipynb) notebook.

In [ ]:
%pip install -U -q "google-genai>=2.9.0" # 2.0 is needed to use the interactions API

### Setup your API key

To run the following cell, your API key must be stored in a Colab Secret named `GEMINI_API_KEY`. If you don't already have an API key, or you're not sure how to create a Colab Secret, see [Authentication ![image](https://storage.googleapis.com/generativeai-downloads/images/colab_icon16.png)](../quickstarts/Authentication.ipynb) for a walkthrough.

In [13]:
from google.colab import userdata

GEMINI_API_KEY=userdata.get('GEMINI_API_KEY')

### Initialize SDK client

With the new SDK you now only need to initialize a client with your API key (or OAuth if using [Vertex AI](https://cloud.google.com/vertex-ai)). The model is now set in each call.

In [15]:
from google import genai
from google.genai import types

client = genai.Client(api_key=GEMINI_API_KEY)

In [16]:
MODEL_ID = "gemini-3.6-flash" # @param ["gemini-3.5-flash-lite", "gemini-2.5-flash", "gemini-3.6-flash", "gemini-2.5-pro", "gemini-2.5-flash-preview", "gemini-3-flash-preview", "gemini-3.1-pro-preview"] {"allow-input":true, isTemplate: true}

### Imports

In [18]:
import json
from PIL import Image
from IPython.display import display, Markdown

## Using thinking models

Here are some complex examples of what Gemini thinking models can solve.

In each of them you can select different models to see how they compare.

In some cases, you'll still get a good answer even with lower thinking levels. In that case, re-run the example with `minimal` thinking to see how much the model benefits from deeper reasoning.

### Using thinking levels

You can start by asking the model to explain a concept and see how it does reasoning before answering.

By default, the model uses the `high` thinking level, which lets it dynamically adjust its reasoning depth based on the complexity of the request.

In [21]:
prompt = """
    You are playing the 20 question game. You know that what you are looking for
    is a aquatic mammal that doesn't live in the sea, is venomous and that's
    smaller than a cat. What could that be and how could you make sure?
"""

interaction = client.interactions.create(
    model=MODEL_ID,
    input=prompt
)

Markdown(interaction.steps[-1].content[0].text)

Looking to the response metadata, you can see not only the amount of tokens on your input and the amount of tokens used for the response, but also the amount of tokens used for the thinking step - As you can see here, the model used around 1400 tokens in the thinking steps:

In [23]:
print("Prompt tokens:",response.usage_metadata.prompt_token_count)
print("Thoughts tokens:",response.usage_metadata.thoughts_token_count)
print("Output tokens:",response.usage_metadata.candidates_token_count)
print("Total tokens:",response.usage_metadata.total_token_count)

Prompt tokens: 62
Thoughts tokens: 710
Output tokens: 522
Total tokens: 1294


### Minimal thinking

You can set thinking to `minimal` to get faster responses with almost no reasoning. You'll see that in this case, the model doesn't think through the problem as deeply.

In [25]:
if "-pro" not in MODEL_ID:
  prompt = """
      You are playing the 20 question game. You know that what you are looking for
      is a aquatic mammal that doesn't live in the sea, is venomous and that's
      smaller than a cat. What could that be and how could you make sure?
  """

  interaction = client.interactions.create(
    model=MODEL_ID,
    input=prompt,
    config=types.GenerateContentConfig(
      thinking_config=types.ThinkingConfig(
        thinking_level="none"
      )
    )
  )

  Markdown(interaction.steps[-1].content[0].text)

else:
  print("You can't disable thinking for pro models.")

Now you can see that the response is faster as the model didn't perform any thinking step. Also you can see that no tokens were used for the thinking step:

In [27]:
print("Prompt tokens:",response.usage_metadata.prompt_token_count)
print("Thoughts tokens:",response.usage_metadata.thoughts_token_count)
print("Output tokens:",response.usage_metadata.candidates_token_count)
print("Total tokens:",response.usage_metadata.total_token_count)

Prompt tokens: 63
Thoughts tokens: None
Output tokens: 393
Total tokens: 456


<a name="physics"></a>
### Solving a physics problem

Now, try with a complex physics problem. First with `minimal` thinking to see how the model performs:

In [29]:
if "-pro" not in MODEL_ID:
    prompt = """
            A cantilever beam of length L=3m has a rectangular cross-section (width b=0.1m, height h=0.2m) and is made of steel (E=200 GPa).
            It is subjected to a uniformly distributed load w=5 kN/m along its entire length and a point load P=10 kN at its free end.
            Calculate the maximum bending stress (σ_max).
    """

    interaction = client.interactions.create(
            model=MODEL_ID,
            input=prompt,
            config=types.GenerateContentConfig(
                    thinking_config=types.ThinkingConfig(
                            thinking_level="none"
                    )
            )
    )

    Markdown(interaction.steps[-1].content[0].text)

else:
    print("You can't disable thinking for pro models.")

You can see that with minimal thinking, the model uses very few tokens for reasoning:

In [31]:
print("Prompt tokens:",response.usage_metadata.prompt_token_count)
print("Thoughts tokens:",response.usage_metadata.thoughts_token_count)
print("Output tokens:",response.usage_metadata.candidates_token_count)
print("Total tokens:",response.usage_metadata.total_token_count)

Prompt tokens: 97
Thoughts tokens: None
Output tokens: 735
Total tokens: 832


Then you can use `high` thinking to see how the model performs with full reasoning.

You can see that, even for the same prompt, the depth and consistency of the answer improves significantly when the model is allowed to think more deeply.

**NOTE:** You can always see how many tokens were used for the thinking step in the `usage_metadata`.

In [33]:
prompt = """
    A cantilever beam of length L=3m has a rectangular cross-section (width b=0.1m, height h=0.2m) and is made of steel (E=200 GPa).
    It is subjected to a uniformly distributed load w=5 kN/m along its entire length and a point load P=10 kN at its free end.
    Calculate the maximum bending stress (σ_max).
"""

interaction = client.interactions.create(
    model=MODEL_ID,
    input=prompt,
    config=types.GenerateContentConfig(
        thinking_config=types.ThinkingConfig(
            thinking_level=types.ThinkingLevel.HIGH
        )
    )
)

Markdown(interaction.steps[-1].content[0].text)

Now you can see that the model used significantly more tokens for reasoning:

In [35]:
print("Prompt tokens:",response.usage_metadata.prompt_token_count)
print("Thoughts tokens:",response.usage_metadata.thoughts_token_count)
print("Output tokens:",response.usage_metadata.candidates_token_count)
print("Total tokens:",response.usage_metadata.total_token_count)

Prompt tokens: 96
Thoughts tokens: 1116 / 4096
Output tokens: 758
Total tokens: 1970


Keep in mind that higher thinking levels mean the model will spend more time reasoning, which means longer computation time and a more expensive request.

<a name="geometry"></a>
### Working with multimodal problems

This geometry problem requires complex reasoning and is also using Gemini multimodal abilities to read the image.

In [38]:
!wget https://storage.googleapis.com/generativeai-downloads/images/geometry.png -O geometry.png -q

geometry_image = Image.open("geometry.png").resize((256,256))
geometry_image

In [39]:
prompt = "What's the area of the overlapping region?"

interaction = client.interactions.create(
    model=MODEL_ID,
    input=[geometry_image, prompt],
    config=types.GenerateContentConfig(
        thinking_config=types.ThinkingConfig(
            thinking_level=types.ThinkingLevel.HIGH
        )
    )
)

Markdown(interaction.steps[-1].content[0].text)

### Solving brain teasers

Here's another brain teaser based on an image, this time it looks like a mathematical problem, but it cannot actually be solved mathematically. If you check the thoughts of the model you'll see that it will realize it and come up with an out-of-the-box solution.

In [41]:
!wget https://storage.googleapis.com/generativeai-downloads/images/pool.png -O pool.png -q

pool_image = Image.open("pool.png").resize((256,256))
pool_image

First you can check how the model performs with `minimal` thinking:

In [43]:
interaction = client.interactions.create(
    model=MODEL_ID,
    input=[
        pool_image,
        "How do I use those three pool balls to sum up to 30?"
    ],
    config=types.GenerateContentConfig(
        thinking_config=types.ThinkingConfig(
            thinking_level=types.ThinkingLevel.MINIMAL
        )
    )
)

Markdown(interaction.steps[-1].content[0].text)

As you can notice, the model struggled to find a way to get to the result — and ended up suggesting to use different pool balls.

Now you can use `high` thinking to solve the riddle:

In [45]:
prompt = "How do I use those three pool balls to sum up to 30?"

interaction = client.interactions.create(
    model=MODEL_ID,
    input=[
        pool_image,
        prompt,
    ],
    config=types.GenerateContentConfig(
        thinking_config=types.ThinkingConfig(
            thinking_level=types.ThinkingLevel.HIGH
        )
    )
)

Markdown(interaction.steps[-1].content[0].text)

<a name="math"></a>
### Solving a math puzzle with high thinking

This is typically a case where you want to use `high` thinking, as the model needs to explore many directions before finding the right answer.

In [47]:
prompt = """
   How can you obtain 565 with 10 8 3 7 1 and 5 and the common operations?
   You can only use a number once.
"""

interaction = client.interactions.create(
    model=MODEL_ID,
    input=prompt,
    config=types.GenerateContentConfig(
        thinking_config=types.ThinkingConfig(
            thinking_level=types.ThinkingLevel.HIGH
        )
    )
)

display(Markdown(interaction.steps[-1].content[0].text))

<IPython.core.display.Markdown object>


<a name="thoughts_summaries"></a>
## Working thoughts summaries

Summaries of the model's thinking reveal its internal problem-solving pathway. Users can leverage this feature to check the model's strategy and remain informed during complex tasks.

For more details about Gemini thinking capabilities, take a look at the [Gemini models thinking guide](https://googledevai.devsite.corp.google.com/gemini-api/docs/thinking#summaries).

In [49]:
prompt = """
  Alice, Bob, and Carol each live in a different house on the same street: red, green, and blue.
  The person who lives in the red house owns a cat.
  Bob does not live in the green house.
  Carol owns a dog.
  The green house is to the left of the red house.
  Alice does not own a cat.
  Who lives in each house, and what pet do they own?
"""

interaction = client.interactions.create(
  model=MODEL_ID,
  input=prompt,
  config=types.GenerateContentConfig(
    thinking_config=types.ThinkingConfig(
      include_thoughts=True
    )
  )
)

You can check both the thought summaries and the final model response:

In [51]:
for step in interaction.steps:
    if step.type == "model_output":
        for content in step.content:
            if hasattr(content, 'thought') and content.thought:
                display(Markdown("## **Thoughts summary:**"))
                display(Markdown(content.text))
                print()
            elif content.text:
                display(Markdown("## **Answer:**"))
                display(Markdown(content.text))

<IPython.core.display.Markdown object>
<IPython.core.display.Markdown object>

<IPython.core.display.Markdown object>
<IPython.core.display.Markdown object>


You can also use see the thought summaries in streaming experiences:

In [53]:
prompt = """
    Alice, Bob, and Carol each live in a different house on the same street: red,
    green, and blue.
    The person who lives in the red house owns a cat.
    Bob does not live in the green house.
    Carol owns a dog.
    The green house is to the left of the red house.
    Alice does not own a cat.
    Who lives in each house, and what pet do they own?
"""

thoughts = ""
answer = ""

for event in client.interactions.create(
    model=MODEL_ID,
    input=prompt,
    config=types.GenerateContentConfig(
        thinking_config=types.ThinkingConfig(
            thinking_level=types.ThinkingLevel.HIGH
        )
    ),
    stream=True,
):
    if hasattr(event, 'delta'):
        if hasattr(event.delta, 'thought') and event.delta.thought:
            thoughts += event.delta.text or ""
        elif hasattr(event.delta, 'text') and event.delta.text:
            answer += event.delta.text

display(Markdown("## **Thoughts summary:**"))
display(Markdown(thoughts))
print()
display(Markdown("## **Answer:**"))
display(Markdown(answer))

<IPython.core.display.Markdown object>
<IPython.core.display.Markdown object>
<IPython.core.display.Markdown object>
<IPython.core.display.Markdown object>
<IPython.core.display.Markdown object>
<IPython.core.display.Markdown object>
<IPython.core.display.Markdown object>
<IPython.core.display.Markdown object>
<IPython.core.display.Markdown object>
<IPython.core.display.Markdown object>
<IPython.core.display.Markdown object>
<IPython.core.display.Markdown object>
<IPython.core.display.Markdown object>
<IPython.core.display.Markdown object>
<IPython.core.display.Markdown object>
<IPython.core.display.Markdown object>
<IPython.core.display.Markdown object>
<IPython.core.display.Markdown object>
<IPython.core.display.Markdown object>
<IPython.core.display.Markdown object>
<IPython.core.display.Markdown object>
<IPython.core.display.Markdown object>
<IPython.core.display.Markdown object>
<IPython.core.display.Markdown object>
<IPython.core.display.Markdown object>
<IPython.core.display.Mar

## Working with Gemini thinking models and tools

Gemini thinking models are compatible with the tools and capabilities inherent to the Gemini ecosystem. This compatibility allows them to interface with external environments, execute computational code, or retrieve real-time data, subsequently incorporating such information into their analytical framework and concluding statements.

<a name="code_execution"></a>
### Solving a problem using the code execution tool

This example shows how to use the [code execution](./Code_Execution.ipynb) tool to solve a problem. The model will generate the code and then execute it to get the final answer.

In [56]:
prompt = """
    What are the best ways to sort a list of n numbers from 0 to m?
    Generate and run Python code for three different sort algorithms.
    Provide the final comparison between algorithm clearly.
    Is one of them linear?
"""

code_execution_tool = types.Tool(
    code_execution=types.ToolCodeExecution()
)

interaction = client.interactions.create(
    model=MODEL_ID,
    input=prompt,
    config=types.GenerateContentConfig(
        tools=[code_execution_tool],
        thinking_config=types.ThinkingConfig(
            thinking_level=thinking_level,
        )
    ),
)

Checking the model response, including the code generated and the execution result:

In [58]:
from IPython.display import display, HTML, Markdown

for step in interaction.steps:
    if step.type == "model_output":
        for content in step.content:
            if content.text is not None:
                display(Markdown(content.text))
    elif step.type == "code_execution":
        display(HTML(f'<pre style="background-color: #1a1a2e; color: #16c60c; padding: 10px;">{step.code}</pre>'))
        if hasattr(step, 'result') and step.result:
            display(HTML(f'<pre style="background-color: #2d2d44; color: white; padding: 10px;">{step.result}</pre>'))

<IPython.core.display.HTML object>
<IPython.core.display.Markdown object>
<IPython.core.display.Markdown object>
<IPython.core.display.Markdown object>


<a name="google_search"></a>
### Thinking with search tool

Search grounding is a great way to improve the quality of the model responses by giving it the ability to search for the latest information using Google Search. Check the [dedicated guide](./Search_Grounding.ipynb) for more details on that feature.

In [60]:
from google.genai.types import Tool, GenerateContentConfig, GoogleSearch

google_search_tool = Tool(google_search=GoogleSearch())

prompt = """
    What were the major scientific breakthroughs announced last month? Use your
    critical thinking and only list what's really incredible and not just an
    overinfluated title.
"""

interaction = client.interactions.create(
    model=MODEL_ID,
    input=prompt,
    config=GenerateContentConfig(
        tools=[google_search_tool],
        thinking_config=types.ThinkingConfig(
            thinking_level=thinking_level,
            include_thoughts=True
        )
    )
)

Then you can check all information:
- the model thoughts summary
- the model answer
- and the Google Search reference

In [62]:
for step in interaction.steps:
    if step.type == "model_output":
        for content in step.content:
            if hasattr(content, 'thought') and content.thought:
                display(Markdown("## **Thoughts summary:**"))
                display(Markdown(content.text))
                print()
            elif content.text:
                display(Markdown("## **Answer:**"))
                display(Markdown(content.text))

<IPython.core.display.Markdown object>
<IPython.core.display.Markdown object>

<IPython.core.display.Markdown object>
<IPython.core.display.Markdown object>

<IPython.core.display.Markdown object>
<IPython.core.display.Markdown object>

<IPython.core.display.Markdown object>
<IPython.core.display.Markdown object>
<IPython.core.display.Markdown object>
<IPython.core.display.HTML object>


<a name="thinking_level"></a>
## Thinking levels reference

Thinking levels provide a simple way to control the amount of reasoning:

```python
# Set thinking level
config = types.GenerateContentConfig(
    thinking_config=types.ThinkingConfig(
        thinking_level=types.ThinkingLevel.HIGH  # MINIMAL, LOW, MEDIUM, HIGH
    )
)
```

In [ ]:
# @title Run this cell to set everything up if you jump directly to this section
from google.colab import userdata
from google import genai
from google.genai import types
from IPython.display import display, Markdown, HTML

client = genai.Client(api_key=userdata.get('GEMINI_API_KEY'))

# Select the Gemini 3 model

GEMINI_3_MODEL_ID = "gemini-3.6-flash" # @param ["gemini-3.5-flash-lite", "gemini-3.6-flash", "gemini-3-flash-preview", "gemini-3.1-pro-preview"] {"allow-input":true, isTemplate: true}

You can set the thinking level to `minimal`, `low`, `medium` or `high` (default). This will indicate to the model how much thinking it is allowed to do. Since the thinking process stays dynamic, `high` doesn't necessarily mean the model will think a lot — it just means it's allowed to think as much as it needs.

Similarly, setting the level to `low` doesn't mean the model will think very little, it means it'll be restricted and might not think as much as it should for more complex tasks.

In [66]:
prompt = """
  Find what I'm thinking of:
    It moves, but doesn't walk, run, or swim.
    It has no fixed shape and if cut into pieces, those pieces will keep living and moving.
    It has no brain but can solve complex mazes.
"""

# Thinking levels can be either "Minimal/Low/Medium/High" or types.ThinkingLevel.MINIMAL/types.ThinkingLevel.LOW/types.ThinkingLevel.MEDIUM/types.ThinkingLevel.HIGH
thinking_level = "High" # @param ["Minimal", "Low", "Medium","High"]

interaction = client.interactions.create(
  model=GEMINI_3_MODEL_ID,
  input=prompt,
  config=types.GenerateContentConfig(
    thinking_config=types.ThinkingConfig(
      thinking_level=thinking_level,
      include_thoughts=True
    )
  )
)

for part in response.parts:
  if not part.text:
    continue
  if part.thought:
    display(Markdown("### Thought summary:"))
    display(Markdown(part.text))
    print()
  else:
    display(Markdown("### Answer:"))
    display(Markdown(part.text))
    print()

print(f"We used {response.usage_metadata.thoughts_token_count} tokens for the thinking phase and {response.usage_metadata.prompt_token_count} for the output.")

<IPython.core.display.Markdown object>
<IPython.core.display.Markdown object>

<IPython.core.display.Markdown object>
<IPython.core.display.Markdown object>

We used 282 tokens for the thinking phase and 62 for the output.


<a name="gemini3migration"></a>
### Legacy: Migrating from `thinking_budget` to `thinking_level`

If you were previously using `thinking_budget` (a feature from `generateContent`), here's how to migrate:

* If you were using `thinking_budget=0` → use `ThinkingLevel.MINIMAL`
* If you were using a low `thinking_budget` (e.g., 1024) → use `ThinkingLevel.LOW`
* If you were using a medium `thinking_budget` (e.g., 4096-8192) → use `ThinkingLevel.MEDIUM`
* If you were using the maximum `thinking_budget` or adaptive (default) → use `ThinkingLevel.HIGH`
* If you were using **dynamic thinking** → use `ThinkingLevel.HIGH` (the model will still adjust dynamically)

> **Note:** `thinking_budget` is a legacy feature from the `generateContent` API that can still be used to fine-tune thinking behavior. See the [generateContent notebook](./Get_started_Generate_Content.ipynb) for details.

# Next Steps

Try Gemini 3 and other Gemini models in
[Google AI Studio](https://aistudio.google.com/prompts/new_chat?model=gemini-3.6-flash), and learn more about [Prompting for thinking models](https://ai.google.dev/gemini-api/docs/thinking#best-practices).

For more examples of the Gemini capabilities, check the other [Cookbook examples](https://github.com/google-gemini/cookbook). You'll learn how to use the [Live API ![image](https://storage.googleapis.com/generativeai-downloads/images/colab_icon16.png)](./Get_started.ipynb), juggle with [multiple tools ![image](https://storage.googleapis.com/generativeai-downloads/images/colab_icon16.png)](../examples/LiveAPI_plotting_and_mapping.ipynb) or use Gemini [spatial understanding ![image](https://storage.googleapis.com/generativeai-downloads/images/colab_icon16.png)](./Spatial_understanding.ipynb) abilities.